# 02 — Boundary ground truth extraction

**What this notebook does.** Turns each dataset's masks into boundary ground
truth: a uint8 PNG per pair whose pixels are exactly 0 or 255, one uniform
line width, written to `PERSISTENT_DIR/gt_boundaries/<folder>/`. Which
extractor runs is decided per folder by `reports/audit.json` — HSV colour
thresholding for MODE A, `find_boundaries` on a snapped label map for MODE B —
with an override dict for a verdict a human disagrees with.

**What must already exist.**

- `GH_TOKEN` as a host secret and the datasets mounted, i.e.
  `notebooks/00_bootstrap.ipynb` passes
- `reports/audit.json` in the repo, i.e. `notebooks/01_audit.ipynb` has run
  and pushed. This notebook reads every folder's mode from it and never guesses.

**What it produces.** Boundary PNGs under `PERSISTENT_DIR/gt_boundaries/`
(persistent, not committed — they are derived data), plus
`configs/hsv_ranges.yaml` and `reports/gt_extraction.{md,json}`, which the
last cell pushes back to the repo.

**Expected runtime on a free T4.** 15–30 minutes for all five folders (~1500
pairs), dominated by Steel1's 907 pairs. No GPU is used: the whole path is
NumPy, OpenCV and scikit-image morphology. Set `LIMIT` in the extraction cell
to a small number for a 1-minute smoke test first.

**What this step cannot do.** Every folder in this collection is MODE B, so
the output is *phase interfaces only*. Grain boundaries between two grains of
the same phase are absent from the source masks and are not invented here —
that would put a learned model in the label path, which this project forbids.

## Cell 1 — the standard bootstrap block

Identical in every notebook. Reads `GH_TOKEN` from the host secret store,
fetches `scripts/bootstrap_session.py` through the GitHub API, then hands over
to `bootstrap()`, which clones the repo, installs what is missing, mounts
Drive on Colab, verifies `DATA_ROOT` and returns `PATHS`. Re-running it after
a disconnect is the correct way to recover.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## The mode assignment being used

Step 1 measured each folder's mask kind and wrote it to `reports/audit.json`.
This step obeys that file. `MODE_OVERRIDES` below is the escape hatch: set
`{"uhcs2": "A"}` to force a folder through the other extractor when you
believe the audit is wrong. Any override is recorded in
`reports/gt_extraction.md`, so a run can never silently disagree with its audit.

The cell also prints what MODE B will actually quantize to. **K is the number
of colour *peaks*, not the raw unique-colour count.** Steel2's masks report
45–52 colours where there are two real classes: the rest is an anti-aliasing
ramp decaying smoothly away from black and white. Colours within
`palette_merge_distance` (48 in RGB) of a heavier colour are treated as ramp
values of that colour and snap to it; genuine classes in this collection are
more than 190 apart, so the margin is wide.

Pairs listed under `exclusions` are never processed at all. `uhcs0596.png` in
uhcs2 is a 645×484 mask of a single flat colour — no annotation whatsoever,
while its micrograph shows a clear boundary network. That is bad data, not a
processing failure, so it is excluded here and reported as excluded rather
than appearing as a rejection later.

In [ ]:
import json
from pathlib import Path

from src import boundary_gt

# ---- override a verdict here if you disagree with the audit --------------
# folder -> "A" or "B".  Empty dict = use reports/audit.json exactly.
MODE_OVERRIDES = {}
# e.g. MODE_OVERRIDES = {"uhcs2": "A"}

reports_dir = Path(PATHS["reports_dir"])
audit = boundary_gt.load_audit(reports_dir)
settings = boundary_gt.load_config()
modes = boundary_gt.folder_modes(audit, MODE_OVERRIDES)

print(f"audit generated {audit['generated_utc']}")
print(f"settings        {json.dumps(settings, sort_keys=True)}\n")

for name, mode in modes.items():
    d = audit["datasets"][name]
    tag = ""
    if name in MODE_OVERRIDES:
        tag = f"   <-- OVERRIDDEN (audit said {d['mode']['inferred']})"
    print(f"{name:<12} MODE {mode}{tag}")
    print(f"{'':<12} audit reason: {d['mode']['reason']}")
    if mode == "B":
        palette = boundary_gt.derive_palette(d, settings)
        raw = d["palette"]["n_unique_colours"]["median"]
        print(f"{'':<12} K = {len(palette)} peaks {palette}")
        print(f"{'':<12} (raw unique colours per mask, median: {raw:.0f})")
    else:
        print(f"{'':<12} boundary colour from audit: "
              f"{boundary_gt.boundary_colour(d)}")
    excl = (settings["exclusions"] or {}).get(name, [])
    if excl:
        print(f"{'':<12} excluded, never processed: {excl}")
    watch = boundary_gt.watched_colours(name, settings)
    if watch:
        folding = "FOLDED into background" if settings["fold_artifact_colours"] \
            else "watched only, kept as its own class"
        print(f"{'':<12} artifact colours: {[list(c) for c in watch]} -- {folding}")
    print()

## What HSV thresholding does, and how to fix it when it is wrong

*This section applies to MODE A folders. On this collection there are none —
every dataset is a phase-label map — so the cell below will say so and skip.
It is here because the moment a MODE A dataset arrives, or you override a
verdict above, this is the machinery that runs, and it is the part that needs
a human eye.*

RGB is a bad space to select a painted colour in: change the brightness of a
green line and all three channels move together, so no fixed box in RGB holds
"green" across a dataset. HSV separates *which* colour (hue, 0–179 in
OpenCV's 8-bit convention) from *how saturated* (0–255) and *how bright*
(0–255). A painted annotation is a narrow band of hue with high saturation, so
a box in HSV — `cv2.inRange(hsv, lower, upper)` — selects it robustly.

**The window is derived, never hardcoded.** `derive_hsv_window` collects the
pixels that actually carry the boundary colour the audit identified, across up
to `hsv_sample_files` masks, and takes the 2nd and 98th percentile of H, S and
V. The 2/98 trim is what keeps a handful of JPEG-fringed or anti-aliased
pixels from stretching the window over half the colour wheel. Red is the one
special case: its hue straddles the 0/179 wrap, which is detected and emitted
as two windows OR-ed together.

**When the result is wrong:**

- *Boundaries come out broken, dotted, or thinner than they look in the mask* —
  the window is too tight. Widen it: raise the percentile spread (2/98 → 1/99),
  or edit `configs/hsv_ranges.yaml` and enlarge the S and V range first, since
  compression noise moves saturation and brightness far more than hue.
- *Whole phase regions bleed into the boundary map* — the window is too wide.
  Tighten hue first: a painted line usually sits within ±5 hue units of its
  nominal colour, and it is the S/V floor that is letting a dark or washed-out
  phase in. Raise the lower S bound.
- *One folder needs different numbers from the rest* — edit its entry in
  `configs/hsv_ranges.yaml` and set `manual: true` on it. A window marked
  manual is never overwritten by a later extraction run.

The grid below shows the same mask thresholded at half, one and double the
derived window width, so you can see which way the error goes before touching
anything.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src import audit as audit_mod

mode_a_folders = [n for n, m in modes.items() if m == "A"]

if not mode_a_folders:
    print("No MODE A folder in this run: every folder in reports/audit.json is "
          "a phase-label map, so no HSV window is derived and none is needed.")
    print("Set MODE_OVERRIDES above to force a folder through MODE A to see "
          "this grid.")
else:
    for name in mode_a_folders:
        d = audit["datasets"][name]
        pairing = audit_mod.discover_pairs(Path(d["path"]))
        window = boundary_gt.derive_hsv_window(d, pairing["pairs"], settings)

        print(f"{name}: boundary colour {window['colour_rgb']}, derived from "
              f"{window['n_pixels_sampled']} pixels "
              f"at percentiles {window['percentiles']}")
        print(f"    HSV lower {[round(v, 1) for v in window['lower']]}")
        print(f"    HSV upper {[round(v, 1) for v in window['upper']]}"
              + ("   (hue wraps 0/179)" if window["wraps_hue"] else ""))

        img_path, mask_path = pairing["pairs"][0]
        mask = audit_mod.read_array(mask_path)

        scales = [0.5, 1.0, 2.0]
        fig, axes = plt.subplots(1, len(scales) + 1, figsize=(5 * (len(scales) + 1), 5))
        axes[0].imshow(mask, interpolation="nearest")
        axes[0].set_title(f"{name} mask\n{mask_path.name}", fontsize=9)
        for ax, scale in zip(axes[1:], scales):
            w = boundary_gt.scale_hsv_window(window, scale)
            hit = boundary_gt.apply_hsv_window(mask, w)
            ax.imshow(hit, cmap="gray", interpolation="nearest")
            ax.set_title(f"window x{scale}  ->  {hit.mean():.4f} of pixels\n"
                         f"H {w['lower'][0]:.0f}-{w['upper'][0]:.0f}  "
                         f"S {w['lower'][1]:.0f}-{w['upper'][1]:.0f}  "
                         f"V {w['lower'][2]:.0f}-{w['upper'][2]:.0f}", fontsize=9)
        for ax in axes:
            ax.set_xticks([])
            ax.set_yticks([])
        plt.tight_layout()
        plt.show()

## Run the extraction

All of the work is in `src/boundary_gt.py`; this cell supplies the paths and a
progress bar. Per pair, whichever extractor its folder's mode selects produces
a raw binary map, and then both modes share one cleanup chain:

1. **Speckle removal (`open_kernel` 3)** — drops connected components smaller
   than 9 px: single-pixel label noise in MODE B, compression fringe in MODE A.
   This is what a 3×3 morphological OPEN was meant to do, but **not** how it is
   done, and the difference matters. A literal OPEN erodes first, keeping a
   pixel only when all nine of its neighbours are set — and the thing being
   cleaned is a line 1–3 px wide (`find_boundaries(mode="thick")` gives 2 px).
   A 3×3 OPEN erases the entire boundary map and leaves a blank image. The
   area-based version removes the same speckle and leaves thin lines alone.
   Set `boundary_gt.open_mode: morph` in `configs/default.yaml` if you want the
   literal operation; expect empty output.
2. **CLOSE 3×3** — dilate then erode: seals 1–2 px gaps so a boundary that was
   nicked by noise stays one connected line.
3. **skeletonize** — collapse whatever width survived to a 1 px centreline, so
   the width of the label no longer depends on how thick the annotation
   happened to be.
4. **dilate to `line_width_px`** — grow that centreline back to one uniform
   width (2 px by default) across every dataset. This matters: a network
   trained on labels whose width varies per folder learns folder identity, not
   boundaries.

Nothing is resized at any point, and the output is written as uint8 PNG with
values strictly in {0, 255}.

Set `LIMIT = 5` for a smoke test over a few pairs per folder before committing
to the full run; `LIMIT = None` processes everything.

In [ ]:
from tqdm.auto import tqdm

LIMIT = None   # e.g. 5 for a quick smoke test, None for everything

# GT_BOUNDARIES_ROOT, resolved independently of PERSISTENT_DIR: on Kaggle
# the boundary maps are a separate read-only input, not something under
# /kaggle/working. Step 2 WRITES here, so it needs a writable host.
out_root = Path(PATHS["gt_boundaries_root"])


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False, unit="pair")


report = boundary_gt.extract_all(
    audit,
    out_root,
    settings,
    mode_overrides=MODE_OVERRIDES,
    limit=LIMIT,
    progress=progress,
)

md_path, json_path = boundary_gt.write_extraction_report(report, reports_dir)
hsv_path = boundary_gt.write_hsv_ranges(report)

print(f"\noutput root: {out_root}\n")
for name, d in report["datasets"].items():
    print(f"{name:<12} MODE {d['mode']}  processed {d['n_processed']:>5}  "
          f"rejected {d['n_rejected']:>3}  excluded {d['n_excluded']:>2}  "
          f"frac before {d['fraction_before']['median'] or 0:.4f}  "
          f"after {d['fraction_after']['median'] or 0:.4f}")
for name, err in report["failures"].items():
    print(f"{name:<12} FAILED  {err}")

print(f"\nwrote {md_path}\nwrote {json_path}\nwrote {hsv_path}")

## QC: six triptychs per folder, plus the artifact-colour check

The numbers cannot tell you whether the boundary map traces the *right* lines.
Each row is one pair: the raw micrograph, the mask as stored, and the boundary
PNG that was just written.

What to look for. The extracted line should sit exactly on the colour
transitions in the mask, be closed rather than dotted, and be the same width
everywhere. In MODE B it traces *only* the borders between differently
coloured regions — if two neighbouring grains carry the same colour, the line
between them is correctly absent, because that boundary is not in the source
data. Suspicious signs: a thick blob instead of a line (cleanup failed), lines
along the image border only (label map has a frame artefact), or a boundary
that ignores an obvious colour edge (that colour pair merged during snapping,
so `palette_merge_distance` is too large).

**The fourth panel — artifact colours.** `find_boundaries` cannot tell a phase
from a scale bar: any colour kept as its own class gets a closed boundary loop
drawn around every region of it. For folders with colours listed under
`boundary_gt.artifact_colours`, a fourth panel shows exactly that — the
colour's own pixels in **magenta**, the boundary pixels lying on its outline in
**cyan**, the rest of the boundary in grey — and the printed line gives the
share of that image's boundary attributable to it.

Read it as a decision, not a verdict. A colour that appears in a handful of
masks and contributes a percent or two of boundary is annotation; a colour
present in every mask, covering a real area, whose loops follow the
microstructure is a phase, and folding it away would delete a genuine
interface. Folding is off by default for exactly this reason: set
`fold_artifact_colours: true` in `configs/default.yaml` only after looking at
these panels, and note that it applies to every folder listed.

In [ ]:
import matplotlib.pyplot as plt

from PIL import Image

N_QC = 6

for name, d in report["datasets"].items():
    if not d["files"]:
        print(f"{name}: nothing processed, skipping QC")
        continue
    pairing = audit_mod.discover_pairs(Path(audit["datasets"][name]["path"]))
    by_image = {img.name: (img, msk) for img, msk in pairing["pairs"]}
    watch = boundary_gt.watched_colours(name, settings)

    # Reconciled pairs go first: they are the ones whose geometry changed, so
    # they are the ones that most need looking at.
    recon = {tuple(r["pair"]) for r in d.get("reconciled", [])}
    rows = ([f for f in d["files"] if tuple(f["pair"]) in recon]
            + [f for f in d["files"] if tuple(f["pair"]) not in recon])[:N_QC]
    n_col = 4 if watch else 3
    fig, axes = plt.subplots(len(rows), n_col, figsize=(4.7 * n_col, 4.4 * len(rows)),
                             squeeze=False)
    folded = "folded into background" if d["folded_colours"] else "kept as its own class"
    title = (f"{name} — MODE {d['mode']} — raw | mask | extracted boundary "
             f"({settings['line_width_px']} px)")
    if watch:
        title += f" | artifact {[list(c) for c in watch]} {folded}"
    fig.suptitle(title, fontsize=13)

    for r, rec in enumerate(rows):
        img_path, mask_path = by_image[rec["pair"][0]]
        image = audit_mod.read_array(img_path)
        mask = audit_mod.read_array(mask_path)
        bnd = np.asarray(Image.open(Path(d["out_dir"]) / rec["output"]))

        axes[r][0].imshow(image, cmap="gray" if image.ndim == 2 else None,
                          interpolation="nearest")
        axes[r][0].set_title(f"raw · {img_path.name}", fontsize=9)
        axes[r][1].imshow(mask, cmap="gray" if mask.ndim == 2 else None,
                          interpolation="nearest")
        axes[r][1].set_title(f"mask · {mask_path.name}", fontsize=9)
        axes[r][2].imshow(bnd, cmap="gray", interpolation="nearest")
        note = ""
        if rec.get("size_reconciled"):
            sr = rec["size_reconciled"]
            note = (f"\nSIZE-RECONCILED: image "
                    f"{sr['image_size'][0]}x{sr['image_size'][1]}, mask "
                    f"{sr['mask_size'][0]}x{sr['mask_size'][1]} -> "
                    f"{bnd.shape[1]}x{bnd.shape[0]}")
        axes[r][2].set_title(f"boundary · {rec['output']}\n"
                             f"{rec['fraction_before']:.4f} before -> "
                             f"{rec['fraction_after']:.4f} after cleanup" + note,
                             fontsize=9)

        if watch:
            vis = np.zeros(bnd.shape + (3,), dtype=np.uint8)
            vis[bnd > 0] = (90, 90, 90)
            notes = []
            for colour in watch:
                stats, hit, ring = boundary_gt.artifact_boundary_overlap(
                    mask, colour, bnd, return_masks=True)
                if stats["present"]:
                    vis[(bnd > 0) & (ring | hit)] = (0, 255, 255)
                    vis[hit] = (255, 0, 255)
                notes.append(
                    f"{list(colour)}: "
                    + ("absent" if not stats["present"] else
                       f"{stats['fraction']:.4f} of pixels, "
                       f"{stats['share_of_boundary']:.4f} of boundary on its outline")
                )
            axes[r][3].imshow(vis, interpolation="nearest")
            axes[r][3].set_title("artifact colour (magenta) and the boundary\n"
                                 "drawn around it (cyan)", fontsize=9)
            print(f"{name} · {mask_path.name}: " + "; ".join(notes))

        for ax in axes[r]:
            ax.set_xticks([])
            ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    for a in d["artifacts"]:
        print(f"{name} artifact {a['colour']}: present in "
              f"{a['files_present']}/{a['files_checked']} processed files, "
              f"median {a['pixel_fraction']['median'] or 0:.4f} of pixels, "
              f"median {a['share_of_boundary']['median'] or 0:.4f} of the boundary "
              f"on its outline; folded={a['folded_into_background']}")

## Checks

Nothing here runs locally, so this cell is where the step is declared correct.
Per folder it re-opens the files that were actually written — not the
in-memory report — and asserts four things:

- **strictly {0, 255}**: a boundary label with intermediate values would mean a
  smoothed or interpolated write, and would poison a BCE/Dice loss silently.
- **boundary fraction inside 0.5%–25%**: below that the extractor found almost
  nothing; above it, it is selecting regions rather than lines. This is the
  band that separates a boundary map from a failure that still produces a file.
- **rejections are zero or explained**: every rejected pair must carry a
  reason. Exclusions are counted separately — they are data that was never
  supposed to be processed, not a failure of this step.
- **measured line width matches config**: area ÷ skeleton length on the written
  file, which must come back at `line_width_px` ± 0.5. This is what proves the
  dilation step produced one uniform width rather than whatever the annotation
  happened to be.

In [ ]:
import numpy as np
from PIL import Image

CHECK_SAMPLE = 12   # files re-opened per folder

checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


lo_band, hi_band = boundary_gt.SANE_FRACTION_BAND
width_cfg = int(settings["line_width_px"])

check("extraction report written", md_path.is_file() and json_path.is_file(),
      f"{md_path.name}, {json_path.name}")
check("hsv_ranges.yaml written", hsv_path.is_file(), str(hsv_path.name))
check("no folder failed outright", not report["failures"],
      ", ".join(report["failures"]) or "none")

for name, d in report["datasets"].items():
    print(f"\n--- {name} (MODE {d['mode']}) ---")
    out_dir = Path(d["out_dir"])

    check(f"{name}: pairs processed", d["n_processed"] > 0,
          f"{d['n_processed']} of {d['n_pairs']} pairs "
          f"({d['n_excluded']} excluded, {d['n_rejected']} rejected)")

    on_disk = sorted(p for p in out_dir.glob("*.png")) if out_dir.is_dir() else []
    check(f"{name}: one PNG per processed pair",
          len(on_disk) == d["n_processed"],
          f"{len(on_disk)} files in {out_dir}")

    sample = on_disk[:CHECK_SAMPLE]
    values, widths, fracs = set(), [], []
    for path in sample:
        arr = np.asarray(Image.open(path))
        values |= set(np.unique(arr).tolist())
        fracs.append(float((arr > 0).mean()))
        w = boundary_gt.measured_line_width(arr)
        if w is not None:
            widths.append(w)

    check(f"{name}: values strictly 0/255", values <= {0, 255},
          f"found {sorted(values)} over {len(sample)} files")

    med_frac = float(np.median(fracs)) if fracs else 0.0
    check(f"{name}: boundary fraction in {lo_band:.3f}-{hi_band:.2f}",
          lo_band <= med_frac <= hi_band, f"median {med_frac:.4f}")

    med_w = float(np.median(widths)) if widths else None
    check(f"{name}: line width == {width_cfg} px",
          med_w is not None and abs(med_w - width_cfg) <= 0.5,
          f"measured {med_w:.2f} px" if med_w else "no line pixels")

    explained = all(r.get("why") for r in d["rejected"])
    check(f"{name}: rejections are zero or explained",
          d["n_rejected"] == 0 or explained,
          "none" if d["n_rejected"] == 0 else f"{d['n_rejected']} rejected, all with a reason")
    for a in d["artifacts"]:
        share = a["share_of_boundary"]["median"] or 0.0
        print(f"        artifact {a['colour']}: {a['files_present']}/"
              f"{a['files_checked']} files, {share:.4f} of boundary on its "
              f"outline, folded={a['folded_into_background']}")
    if settings["fold_artifact_colours"] and d["mode"] == "B":
        expected = [list(c) for c in boundary_gt.watched_colours(name, settings)]
        check(f"{name}: artifact colours folded away",
              all(a["share_of_boundary"]["median"] in (None, 0.0)
                  for a in d["artifacts"]),
              f"folding ON for {expected}")
    if d.get("n_reconciled"):
        # Prove the crop was APPLIED, not merely tolerated: re-open each
        # written boundary PNG and compare its real dimensions with the
        # common size the reconciliation claims to have cropped to.
        sizes_ok, crops_ok, notes = True, True, []
        for rec in d["reconciled"]:
            out_png = out_dir / (Path(rec["pair"][0]).stem + ".png")
            written = Image.open(out_png).size if out_png.is_file() else None
            sizes_ok &= written is not None and list(written) == rec["final_size"]
            crop = rec.get("image_crop") or {}
            crops_ok &= all(k in crop for k in ("left", "top", "width", "height"))
            notes.append(
                f"{rec['pair'][1]}: image "
                f"{rec['image_size'][0]}x{rec['image_size'][1]}, mask "
                f"{rec['mask_size'][0]}x{rec['mask_size'][1]} -> written "
                + (f"{written[0]}x{written[1]}" if written else "MISSING")
            )
        check(f"{name}: {d['n_reconciled']} reconciled pair(s) written at the "
              "cropped size", sizes_ok, "; ".join(notes))
        check(f"{name}: reconciled pairs carry an image_crop record for step 3",
              crops_ok,
              "; ".join(f"{r['pair'][1]} crop "
                        f"{r['image_crop']['left']},{r['image_crop']['top']},"
                        f"{r['image_crop']['width']},{r['image_crop']['height']}"
                        for r in d["reconciled"]))
    for r in d["rejected"][:5]:
        print(f"        rejected {r['pair'][1]}: {r['why']}")
    for e in d["excluded"]:
        print(f"        excluded {e['pair'][1]}: {e['why']}")

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Push the configs and the report back to the repo

`configs/hsv_ranges.yaml` and `reports/gt_extraction.md` are the artefacts a
later step (and a human) reads: which window was used, what K each folder
quantized to, how many pairs survived and at what boundary fraction. The
boundary PNGs themselves stay in `PERSISTENT_DIR` — they are derived data,
they are large, and `.gitignore` keeps them out of the repo deliberately.

In [ ]:
from scripts.push_results import push_results

push_results(
    "step 2: boundary ground truth extraction",
    paths=PATHS,
    expect=[md_path, json_path, hsv_path],
)